In [1]:
import torch
from utils import load_splits, tokenize_dataset, compute_metrics, init_model
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [6]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

In [7]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

import torch
import gc
from pathlib import Path


def training_loop(ds_train, ds_val, learning_rate, model_name, per_device_train_batch_size,
                  gradient_accumulation_step, seed=SEED):
    print(f"Training with learning rate: {learning_rate}")
    model = trainer = None
    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=seed,
        data_seed=seed,
    )
    try:
        # Seed Python, NumPy, and Torch before the randomly initialized classifier head.
        set_seed(seed)
        model = init_model(model_name)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )
        trainer.train()
        # Persist the final TrainerState, including the full history and training summary.
        trainer.save_state()
        return Path(training_args.output_dir)
    finally:
        trainer = model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [11]:
model = "bert-base-uncased"

per_device_train_batch_size=32

gradient_accumulation_step=1

for learning_rate in [1e-5, 2e-5, 3e-5]:
    ds_train = tokenize_dataset(X_train, y_train, model)
    ds_val =  tokenize_dataset(X_val, y_val, model)     
    training_loop(ds_train, ds_val, learning_rate,
                model, per_device_train_batch_size, gradient_accumulation_step, seed=SEED)



Map: 100%|██████████| 9197/9197 [00:00<00:00, 22830.64 examples/s]


Training with learning rate: 1e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1100.98it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Log Loss,Runtime,Samples Per Second,Steps Per Second
1,0.320981,0.241385,0.900837,0.901446,0.897243,0.898998,0.241385,9.464700,971.716000,30.429000
2,0.197461,0.252440,0.904099,0.907158,0.898839,0.901907,0.252438,11.687100,786.933000,24.642000
3,0.141375,0.257016,0.908775,0.907043,0.908492,0.907705,0.257019,9.273100,991.789000,31.057000
4,0.103451,0.310296,0.911493,0.910729,0.909677,0.910177,0.310297,11.271200,815.970000,25.552000
5,0.075679,0.350012,0.912471,0.912036,0.910299,0.911101,0.350010,11.310300,813.150000,25.463000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Training with learning rate: 2e-05


... (3 outputs left out)

Training with learning rate: 3e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2568.64it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Log Loss,Runtime,Samples Per Second,Steps Per Second
1,0.288508,0.221187,0.911384,0.911849,0.908278,0.909813,0.221186,12.372700,743.329000,23.277000
2,0.145993,0.248479,0.913341,0.915691,0.908858,0.911507,0.248478,8.783200,1047.108000,32.790000
3,0.068577,0.402432,0.908775,0.907129,0.908288,0.907670,0.402423,10.878100,845.458000,26.475000
4,0.032123,0.512550,0.913450,0.914885,0.909619,0.911759,0.512506,11.810900,778.684000,24.384000
5,0.013502,0.575374,0.914972,0.915532,0.911902,0.913462,0.575293,11.933900,770.663000,24.133000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [13]:
import json
from pathlib import Path

model_names = ["bert-base-uncased"]
learning_rates = [1e-5, 2e-5, 3e-5]
Path("logs").mkdir(exist_ok=True)

for model_name in model_names:
    for learning_rate in learning_rates:
        state_path = Path("models") / f"{model_name}_{learning_rate}" / "trainer_state.json"
        if not state_path.is_file():
            continue
        history = json.loads(state_path.read_text())["log_history"]
        training_by_epoch = {
            entry["epoch"]: entry for entry in history if "loss" in entry
        }
        results = []
        for entry in history:
            if "eval_f1_macro" in entry:
                results.append({
                    **training_by_epoch.get(entry["epoch"], {}),
                    **entry,
                    "learning_rate": learning_rate,
                })
        log_path = Path("logs") / f"{model_name}_{learning_rate}.json"
        log_path.write_text(json.dumps(results, indent=4))
        print(f"Wrote {log_path}")


Wrote logs/bert-base-uncased_1e-05.json
Wrote logs/bert-base-uncased_2e-05.json
Wrote logs/bert-base-uncased_3e-05.json
